# 1. Data Acquisition

*Proposal section 3.2.1.* Profiles the raw data used to build the trust-scoring models. The platform has no transaction history yet, so this public card-transaction dataset (13.3M transactions from ~1,200 users, 2010-2019) supplies realistic behaviour patterns that later notebooks turn into synthetic worker and client data.

## Setup

Runs locally or in Google Colab. In Colab it mounts Google Drive, copies this repo's code from GitHub, and points `data/` at **My Drive** (raw files in the Drive root, outputs in `My Drive/processed`) and `models/` at `My Drive/models`, so outputs survive between sessions.

In [1]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_URL = "https://github.com/JamieNguru/169421-escrow-based-wallet-and-payment-system.git"
    if not Path("/content/repo").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "dev", REPO_URL, "/content/repo"], check=True)
    ROOT = Path("/content/repo/trust-scoring-ml")
    DRIVE = Path("/content/drive/MyDrive")
    (DRIVE / "models").mkdir(exist_ok=True)
    for name, target in [("data", DRIVE), ("models", DRIVE / "models")]:
        if not (ROOT / name).exists():
            (ROOT / name).symlink_to(target, target_is_directory=True)
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Files

In [2]:
for path in sorted(DATA_DIR.glob("*.*")):
    print(f"{path.name:<28} {path.stat().st_size / 1e6:>9.1f} MB")

cards_data.csv                     0.5 MB
mcc_codes.json                     0.0 MB
paysim.csv                       493.5 MB
train_fraud_labels.json          159.1 MB
transactions_data.csv           1258.5 MB
users_data.csv                     0.2 MB


## Transactions, users, and cards

In [3]:
import json

import pandas as pd

transactions = pd.read_csv(DATA_DIR / "transactions_data.csv", engine="pyarrow")
users = pd.read_csv(DATA_DIR / "users_data.csv")
cards = pd.read_csv(DATA_DIR / "cards_data.csv")

for name, df in [("transactions", transactions), ("users", users), ("cards", cards)]:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("  ", list(df.columns))

transactions: 13,305,915 rows x 12 columns
   ['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors']
users: 2,000 rows x 14 columns
   ['id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']
cards: 6,146 rows x 13 columns
   ['id', 'client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web']


In [4]:
transactions.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


## Merchant category codes and fraud labels

In [5]:
with open(DATA_DIR / "mcc_codes.json", encoding="utf-8") as f:
    mcc_codes = json.load(f)
print(f"{len(mcc_codes)} merchant category codes, e.g.:")
for code_, description in list(mcc_codes.items())[:5]:
    print(f"  {code_}: {description}")

with open(DATA_DIR / "train_fraud_labels.json", encoding="utf-8") as f:
    fraud_labels = pd.Series(json.load(f)["target"])
print(f"\n{len(fraud_labels):,} transactions have a fraud label; "
      f"{(fraud_labels == 'Yes').mean():.3%} of them are fraud.")

109 merchant category codes, e.g.:
  5812: Eating Places and Restaurants
  5541: Service Stations
  7996: Amusement Parks, Carnivals, Circuses
  5411: Grocery Stores, Supermarkets
  4784: Tolls and Bridge Fees



8,914,963 transactions have a fraud label; 0.150% of them are fraud.


## Transaction history per user

In [6]:
per_client = transactions["client_id"].value_counts()
print(f"Unique users: {per_client.size:,}  |  Unique cards: {transactions['card_id'].nunique():,}")
print(f"Date range: {transactions['date'].min()} to {transactions['date'].max()}")
per_client.describe().round(0)

Unique users: 1,219  |  Unique cards: 4,071


Date range: 2010-01-01 00:01:00 to 2019-10-31 23:59:00


count     1219.0
mean     10915.0
std       5607.0
min        760.0
25%       7224.0
50%       9832.0
75%      13349.0
max      48479.0
Name: count, dtype: float64

## Failed transactions, payment channels, and merchant categories

In [7]:
print(f"Transactions with an error: {transactions['errors'].notna().mean():.2%}\n")
print(transactions["errors"].value_counts().head(8).to_string())

Transactions with an error: 1.59%



errors
Insufficient Balance            130902
Bad PIN                          32119
Technical Glitch                 26271
Bad Card Number                   7767
Bad Expiration                    6161
Bad CVV                           6106
Bad Zipcode                       1126
Bad PIN,Insufficient Balance       293


In [8]:
transactions["use_chip"].value_counts()

use_chip
Swipe Transaction     6967185
Chip Transaction      4780818
Online Transaction    1557912
Name: count, dtype: int64

In [9]:
top_mcc = transactions["mcc"].value_counts().head(10)
top_mcc.rename(index=lambda c: f"{c} {mcc_codes.get(str(c), '')}")

mcc
5411 Grocery Stores, Supermarkets     1592584
5499 Miscellaneous Food Stores        1460875
5541 Service Stations                 1424711
5812 Eating Places and Restaurants     999738
5912 Drug Stores and Pharmacies        772913
4784 Tolls and Bridge Fees             674135
5300 Wholesale Clubs                   601942
4829 Money Transfer                    589140
4121 Taxicabs and Limousines           500662
5814 Fast Food Restaurants             499659
Name: count, dtype: int64

## What this means for the models

The data has amounts, timing, failed transactions, and fraud labels, but no worker/client roles and no jobs, disputes, or escrow releases. As the proposal anticipates, those are simulated from these real behaviour patterns in `03_feature_engineering`. The fraud labels are kept aside for external validation in `05_model_evaluation`.